# Assignment 3

## Interactive Dashboard for Video Game Sales Analysis

## Jelisa Lugo 
### jnlugo@umich.edu

#### **Dataset:** vgsales_enriched.csv

#### **Source:** Raj Aditya Nag. *Video Game Sales Data with Other Data*. Kaggle.

#### **Visualization Library:** Plotly

________________________________________________________________________________________________________________________________________________________

## Introduction

In this project, I use the enriched version of the Video Game Sales dataset, called Video Game Sales Data with Other Data, obtained from Kaggle. I explore how game sales vary across genres, platforms, publishers, and release years. Before creating the dashboard, I cleaned the data and performed exploratory data analysis (EDA) to better understand the patterns in the dataset. I then used those findings to build an interactive dashboard in Plotly that makes the data easier to explore.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display
from ipywidgets import interact

In [ ]:
# If the notebook is running in Google Colab, enable the Colab widget manager and renderer.
import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
    renderer = "colab"
# Or use the normal Jupyter Notebook renderer.    
except ImportError:
    renderer = "notebook"


## Load Dataset

In [ ]:
try:
    # Local file (Jupyter Notebook)
    df = pd.read_csv("vgsales_enriched.csv")
except FileNotFoundError:
    # GitHub file (Google Colab)
    df = pd.read_csv("https://raw.githubusercontent.com/jnlugo/SIADS-5-Assignment-3-/refs/heads/main/vgsales_enriched.csv")

## Initial Data Inspection

In [ ]:
df.shape

In [ ]:
df.info()

The dataset contains 16,719 rows and 19 columns. The variables include both numeric and categorical data related to video game sales, game information, and review scores.

In [ ]:
df.head()

## Data Quality Inspection

Before cleaning the data, I checked for missing values, duplicate records, summary statistics, and variable types to better understand the overall quality of the dataset.

In [ ]:
df.isnull().sum()

While looking through the dataset, I noticed some missing values, mostly in the columns realted to reviews of the games. Because they weren't going to impact my analysis I was doing, I decided to leave them as they were.

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

No duplicate observations were found.

## Data Cleaning

After reviewing the dataset, I found that most of the data was already clean. The main issue was the User_Score column, which contained text values that needed to be converted into numeric values before they could be analyzed.

In [ ]:
df["User_Score"] = df["User_Score"].replace("tbd", np.nan)

In [ ]:
df["User_Score"] = pd.to_numeric(df["User_Score"])

In [ ]:
df["User_Score"].dtype

The User_Score column contained the value "tbd" (to be determined), which prevented the column from being treated as numeric. I replaced those values with NaN and then converted the column to a numeric data type. Since those scores were unavailable rather than wrong, I left the missing values as NaN.

In [ ]:
df["Year"].unique()[:20]

In [ ]:
df["Year"].dtype

I also checked the Year column to see if it needed additional cleaning. It is stored as a float and every value represents a whole calendar year. I left the column unchanged.

## Exploratory Data Analysis (EDA)

Before building the dashboard, I wanted to explore the data to understand the main trends and relationships. This analysis helps me decide which visualizations make the most sense to include in the dashboard.

### Which genres generate the highest global sales?

I started by looking at total global sales for each genre to see which types of games sold the most.

In [ ]:
genre_sales = (df.groupby("Genre")["Global_Sales"].sum().sort_values(ascending=False).reset_index())
genre_sales

In [ ]:
# Created an plotly interactive bar chart to compare total global sales by genre so it would easier to create the dashboard later on.
fig = px.bar(
    genre_sales,
    x="Genre",
    y="Global_Sales",
    title="Global Video Game Sales by Genre"
)

fig.show()

### Genre Sales Analysis

Action games had the highest total global sales, followed by Sports and Shooter games. I was a little surprised to see Adventure games much lower on the list because they are a popular genre. One possible reason is that many action-adventure games are classified as Action instead of Adventure.

In [ ]:
platform_sales = (df.groupby("Platform")["Global_Sales"].sum().sort_values(ascending=True).reset_index())
platform_sales.head(10)

In [ ]:
# Created an interactive horizontal bar chart to compare total global sales by platform.
fig = px.bar(
    platform_sales,
    x="Global_Sales",
    y="Platform",
    orientation="h",
    title="Global Video Game Sales by Platform"
)
fig.show()

### Platform Sales Analysis

The PlayStation 2 generated the highest total global sales, followed by the Xbox 360 and PlayStation 3. I was also surprised that the PlayStation 4 and PC ranked lower than expected. Since this dataset contains historical sales, older platforms like the PlayStation 2 had many more years to build large game libraries and accumulate sales.

In [ ]:
year_sales = (df.groupby("Year")["Global_Sales"].sum().reset_index())
year_sales

In [ ]:
# Created an interactive line chart to show global sales over time.
fig = px.line(
    year_sales,
    x="Year",
    y="Global_Sales",
    title="Global Video Game Sales Over Time"
)

fig.show()

### Sales Trends Over Time

Sales continued to grow through the late 1990s and early 2000s before reaching their highest point in 2008. After that, total sales started to decline. I also noticed that the last few years in the dataset have very low sales, but I believe it's due to incomplete data rather than low sales. 

In [ ]:
score_sales = (df[["Critic_Score", "Global_Sales"]].dropna().corr())
score_sales

In [ ]:
df[["User_Score", "Global_Sales"]].corr()

### Review Scores and Global Sales

I also wanted to see if higher review scores were connected to higher game sales. The Critic scores had a weak positive relationship with global sales (r = 0.245), while user scores had an even weaker relationship (r = 0.088). Based on these findings, review scores alone don't have a strong impact on how well a game sells, there are probably other factors that I cannot see that cause a bigger impact.

In [ ]:
# keeping only the data that has both critic scores and global sales 
scatter_data = (df[["Critic_Score", "Global_Sales"]].dropna())

In [ ]:
# Created an interactive scatter plot to compare critic scores and global sales.
fig = px.scatter(
    scatter_data,
    x="Critic_Score",
    y="Global_Sales",
    title="Critic Scores vs. Global Video Game Sales"
)

fig.show()

In [ ]:
publisher_sales = (df.groupby("Publisher")["Global_Sales"].sum().sort_values(ascending=False))
publisher_sales.head()

### Publisher Sales Analysis

Nintendo had the highest total global sales in the dataset, followed by Electronic Arts and Activision. I was't too surprised because Nintendo has some of the biggest game franchises, like Mario and Pokémon. Electronic Arts and Activision also ranked near the top because of their popular sports and action games.

#### How do video game genres perform across different regions?

In [ ]:
region_sales = (
    df.groupby("Genre")[[
        "NA_Sales",
        "EU_Sales",
        "JP_Sales",
        "Other_Sales"
    ]]
    .sum()
)

region_sales

Before creating the heatmap, I grouped the data by genre and calculated the total sales for each region. This made it easier to compare how genres performed across North America, Europe, Japan, and other regions.

In [ ]:
fig = px.imshow(
    region_sales,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Video Game Sales by Genre and Region"
)

# I Renamed the region labels so they are easier to read.
fig.update_xaxes(
    ticktext=["North America", "Europe", "Japan", "Other"],
    tickvals=[0, 1, 2, 3]
)
    
# I Increased the figure size so the heatmap is easier to view thna using the auto since it was very small with the auto layout.    
fig.update_layout(
    width=900,
    height=600
)

fig.show()

The heatmap made it easier to compare how each genre performed across the different regions. I noticed that Action games had the highest sales in both North America and Europe. I expected Role-Playing games to perform well in Japan because many popular Japanese games and franchises are built around role-playing mechanics. Shooter games also had much stronger sales in North America than in Japan. Overall, North America had the highest sales across most genres, with Europe generally ranking second.

#### How are global sales distributed across publishers and genres?

In [ ]:
treemap_data = (df[['Publisher', 'Genre', 'Global_Sales']].dropna())
treemap_data

The table shows each game's publisher, genres, and global sales. I removed any missing values so the treemap would accuratly group games by publisher and genre with including incomplte data. 

In [ ]:
#Created an interactive treemap to show global sales by publisher and genre
fig = px.treemap(treemap_data, 
                 path = ['Publisher','Genre'],
                 values= 'Global_Sales',
                 title= 'Global Video Game Sales by Publisher and Genre')
fig.show()
                 

I found that Nintendo had the largest share of global sales, followed by Electronic Arts and Activision. The treemap also helped me see which genres contributed the most sales for each publisher.

## EDA Summary

Overall, the analysis gave me a better understanding of the dataset before building the dashboard. I found that Action games had the highest total sales, the PlayStation 2 was the top-selling platform, Nintendo was the leading publisher, and game sales peaked around 2008. I also found that review scores had only a weak relationship with global sales. These results helped me decide which charts and filters to include in the interactive dashboard.

## Visualization Library

For this project, I chose Plotly because I wanted to build an interactive dashboard instead of using static charts. Plotly is a free, open-source data visualization library developed by Plotly Technologies that works well with Python and Jupyter notebooks. It supports interactive features like zooming, hovering, panning, and filtering, making it easier for users to explore the data and identify patterns.

To install Plotly, you can use:

python pip install plotly

I chose Plotly because my dashboard uses a few different visualization types, including bar charts, a line chart, a scatter plot, a heatmap, and a treemap. Plotly supports all of these in one library and allows them to be interactive, which makes it a good fit for exploring different parts of the dataset together.

Another reason I chose Plotly is that it integrates well with Jupyter notebooks, which made it easy to build and demonstrate the dashboard within this assignment. Compared to libraries that mainly create static figures, Plotly focuses on interactive visualizations, making it a good choice for dashboards where users can explore the data on their own.

One limitation of Plotly is that creating dashboards with multiple linked charts requires more code than creating individual static graphs. For larger web applications, Plotly also provides Dash, which can be used to build more advanced interactive dashboards.

## Visualization Techniques

The dashboard uses several different visualization types because each one answers a different question about the data.

#### Bar Chart
I used bar charts to compare total sales across genres, platforms, and publishers because they make it easy to see which categories performed the best.

#### Scatter Plot
I included a scatter plot to see whether games with higher review scores also had higher sales. It helped me quickly see that the relationship between review scores and sales was fairly weak.

#### Heatmap

I used a heatmap to compare video game sales across different genres and regions. The differences in shading make it easier to see which genres perform stronger in North America, Europe, Japan, and other regions and to identify regional sales patterns.

#### Treemap

I used a treemap to explore how video game sales are distributed across publishers and genres. The size of each section represents the amount of sales, which makes it easier to see which publishers and genres make up the largest parts of the market.

#### Dashboard Design
These visualizations each show a different part of the video game sales story. The bar charts compare categories, the line chart shows changes over time, the scatter plot explores the relationship between reviews and sales, the heatmap compares regional patterns, and the treemap shows how sales are distributed across larger groups. In the final dashboard, interactive filters allow users to explore the data while multiple visualizations update together.

### Interactive Dashboard 

After creating the individual visualizations, I combined them into an interactive dashboard using Plotly and ipywidgets. The dashboard allows users to filter the dataset and update multiple visualizations at the same time, making it easier to explore sales trends across different genres, publishers, platforms, and years.

In [ ]:
# Created an interactive dashboard.

print("Interactive Video Game Sales Dashboard")
print("Use the filters below to update all four visualizations.")


def create_dashboard(filtered_df):

    # Calculate total global sales by genre.
    genre_sales = (filtered_df.groupby("Genre")["Global_Sales"].sum().sort_values(ascending=False).reset_index())

    # Calculated total global sales by year.
    year_sales = (filtered_df.groupby("Year")["Global_Sales"].sum().reset_index())

    # Removed missing values for scatter plot.
    scatter_df = filtered_df.dropna(subset=["Critic_Score", "Global_Sales"])

    # Removed missing values for treemap.
    treemap_df = filtered_df.dropna(subset=["Publisher", "Genre", "Global_Sales"])

    # Created a 2 x 2 dashboard.
    db = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            "Global Video Game Sales by Genre",
            "Global Video Game Sales Over Time",
            "Critic Score vs. Global Sales",
            "Global Sales by Publisher and Genre"
        ),
        specs=[
            [{"type": "bar"}, {"type": "scatter"}],
            [{"type": "scatter"}, {"type": "domain"}]
        ]
    )

    # Created bar chart.
    bar_fig = px.bar(
        genre_sales,
        x="Genre",
        y="Global_Sales"
    )

    for chart in bar_fig.data:
        db.add_trace(chart, row=1, col=1)

    # Created line chart.
    line_fig = px.line(
        year_sales,
        x="Year",
        y="Global_Sales"
    )

    for chart in line_fig.data:
        db.add_trace(chart, row=1, col=2)

    # Create scatter plot.
    scatter_fig = px.scatter(
        scatter_df,
        x="Critic_Score",
        y="Global_Sales",
        hover_data=["Name"]
    )

    for chart in scatter_fig.data:
        db.add_trace(chart, row=2, col=1)

    # Created treemap.
    tree_fig = px.treemap(
        treemap_df,
        path=["Publisher", "Genre"],
        values="Global_Sales"
    )

    for chart in tree_fig.data:
        db.add_trace(chart, row=2, col=2)

    # Updated the dashboard layout 
    db.update_layout(
        title="Interactive Video Game Sales Dashboard",
        height=900,
        width=1100,
        showlegend=False
    )

    return db


@interact(
    genre=["All"] + sorted(df["Genre"].dropna().unique().tolist()),
    publisher=["All"] + sorted(df["Publisher"].dropna().unique().tolist()),
    year=widgets.IntRangeSlider(
        value=[int(df["Year"].min()), int(df["Year"].max())],
        min=int(df["Year"].min()),
        max=int(df["Year"].max()),
        step=1,
        description="Year:"
    )
)

def update_dashboard(genre, publisher, year):

    filtered_df = df.copy()

    # Filtered the data by genre
    if genre != "All":filtered_df = filtered_df[filtered_df["Genre"] == genre]

    # Filtered the data by publisher
    if publisher != "All":filtered_df = filtered_df[filtered_df["Publisher"] == publisher]

    # Filtered the data by release year
    filtered_df = filtered_df[(filtered_df["Year"] >= year[0]) & (filtered_df["Year"] <= year[1])]

    create_dashboard(filtered_df).show(renderer=renderer)

The completed dashboard combines my four visualizations into a single interactive display. Users can filter the data by genre, publisher, and release year. Each filter updates all four visualizations simultaneously, making it easier to compare trends across multiple perspectives. 

## Deployment

The completed dashboard is hosted publicly so it can be viewed without installing any software. The source code is available in my GitHub repository, and the dashboard can be accessed using the link below.

GitHub Repository:
https://github.com/jnlugo/SIADS-5-Assignment-3-

Dashboard:

## Conclusion
In this project, I cleaned and explored a video game sales dataset before creating an interactive dashboard with Plotly. The dashboard combines multiple visualization types to help users compare genres, platforms, publishers, review scores, and sales trends over time. Building the dashboard gave me experience creating interactive visualizations and showed how combining several charts can tell a more complete story than using a single graph. Overall, this project demonstrated how interactive dashboards can make large datasets easier to explore by allowing users to filter the data and compare multiple visualizations at the same time.

## Troubleshooting

One issue I encountered was that the User_Score column was stored as text because it contained "tbd" values. I replaced those values with NaN before converting the column to a numeric data type.

Another challenge was deciding how to handle missing review scores. Since those values were not required for every visualization, I left them as missing instead of filling them in.

Another issue was while building the dashboard, I originally displayed each visualization separately. I later combined the charts into a single interactive dashboard using Plotly's make_subplots() so that all visualizations appeared together on one screen.

Another issue I ran into was getting my dashboard to work in Google Colab. The dashboard loaded, but the interactive filters weren't updating the visualizations. After doing some research, I found that I needed to enable the Google Colab widget manager and set Plotly to use the Colab renderer. Once I made those changes, the dashboard worked as expected.Since I was submitting both a Jupyter Notebook and a public Google Colab version, I also added a simple try/except statement. This lets the notebook automatically use the correct settings depending on whether it's running in Jupyter Notebook or Google Colab, so I only had to maintain one version of my code.